# AR Dunning Multi-Agent System

LangChain multi-agent pipeline for Accounts Receivable dunning, built with **Claude Agent Teams**.

## Claude Agent Teams that built this project
| Agent | Responsibility |
|---|---|
| **Tools Agent** | `send_invoice_to_customer`, `update_inquiry_status_database`, `update_erp` |
| **Agent Creation Agent** | `ParseCustomerResponse` and `AccessUpdateERP` LangChain agents |
| **Orchestrator Agent** | `orchestrate()`, `generate_status_report()`, `reset_database()` |

## Processing Flow
```
Input_Emails/*.txt
  └─► ParseCustomerResponse  (gemini-2.5-flash-lite)
        ├─ invoice_request  →  send_invoice_to_customer  →  Output/
        └─ promise_date     →  AccessUpdateERP (gemini-3.1-flash)  →  ERPDirectory/invoices.xlsx
        └─ (both paths)     →  update_inquiry_status_database  →  invoice_inquiry.db
  └─► Move email to Processed_Emails/

After all emails → StatusReport saved to Output/
```

## Google Models Used
- `gemini-2.5-flash` — Orchestrator
- `gemini-2.5-flash-lite` — ParseCustomerResponse agent
- `gemini-3.1-flash` — AccessUpdateERP agent
- `gemini-3.5-flash-lite` — Status reporting

> **Note:** Verify exact model IDs at https://ai.google.dev/gemini-api/docs/models before running.

In [1]:
# Uncomment and run once to install dependencies
# !pip install langchain langchain-google-genai google-generativeai pandas openpyxl fpdf2

In [1]:
import os
import json
import shutil
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd
from fpdf import FPDF
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

In [2]:
# ── Google API Key ─────────────────────────────────────────────
# Set via:  export GOOGLE_API_KEY=your-key          (Linux/Mac)
#           $env:GOOGLE_API_KEY = "your-key"        (PowerShell)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "your-google-api-key-here")

# ── Google Gemini Models ────────────────────────────────────────
# Verify current IDs at https://ai.google.dev/gemini-api/docs/models
MODELS = {
    "orchestrator": "gemini-2.5-flash",       # main orchestration
    "parse_agent":  "gemini-2.5-flash-lite",  # ParseCustomerResponse
    "erp_agent":    "gemini-3.1-flash",       # AccessUpdateERP
    "report":       "gemini-3.5-flash-lite",  # status reporting
}

# ── Paths ───────────────────────────────────────────────────────
BASE_DIR             = Path(".")
INPUT_EMAILS_DIR     = BASE_DIR / "Input_Emails"
PROCESSED_EMAILS_DIR = BASE_DIR / "Processed_Emails"
INPUT_INVOICES_DIR   = BASE_DIR / "Input" / "Invoices"
OUTPUT_DIR           = BASE_DIR / "Output"
ERP_DIR              = BASE_DIR / "ERPDirectory"
DB_PATH              = BASE_DIR / "invoice_inquiry.db"
ERP_EXCEL_PATH       = ERP_DIR / "invoices.xlsx"

In [3]:
def setup_directories():
    for d in [INPUT_EMAILS_DIR, PROCESSED_EMAILS_DIR, INPUT_INVOICES_DIR, OUTPUT_DIR, ERP_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print("Directories ready.")


def setup_database():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS invoice_inquiry (
            id             INTEGER PRIMARY KEY AUTOINCREMENT,
            invoice_number TEXT NOT NULL,
            customer_name  TEXT,
            customer_email TEXT,
            inquiry_type   TEXT NOT NULL,
            promise_date   TEXT,
            status         TEXT DEFAULT 'processed',
            email_file     TEXT,
            notes          TEXT,
            processed_at   TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()
    print("Database ready.")


setup_directories()
setup_database()

Directories ready.
Database ready.


In [4]:
# ── Invoice master data (shared by PDF creator and ERP seeder) ──
INVOICE_DATA = [
    {"number": "INV-001", "customer": "ACME Corp",         "email": "john.smith@acmecorp.com",       "amount": 5000.00,  "date": "2026-03-01", "due": "2026-04-01"},
    {"number": "INV-002", "customer": "Tech Startup Inc",  "email": "sarah.jones@techstartup.io",    "amount": 12500.00, "date": "2026-03-15", "due": "2026-04-15"},
    {"number": "INV-003", "customer": "Global Retail Ltd", "email": "mike.chen@globalretail.com",    "amount": 3250.00,  "date": "2026-04-01", "due": "2026-05-01"},
    {"number": "INV-004", "customer": "Manufacturing Net", "email": "emily.white@manufacturing.net", "amount": 8750.00,  "date": "2026-03-20", "due": "2026-04-20"},
    {"number": "INV-005", "customer": "Logistics Co",      "email": "david.brown@logistics.co",      "amount": 15200.00, "date": "2026-04-10", "due": "2026-05-10"},
]


def create_sample_invoices():
    """Create simple PDF invoices in Input/Invoices if they do not already exist."""
    for inv in INVOICE_DATA:
        pdf_path = INPUT_INVOICES_DIR / f"{inv['number']}.pdf"
        if pdf_path.exists():
            continue
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 20)
        pdf.cell(0, 15, "INVOICE", ln=True, align="C")
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(5)
        pdf.set_font("Helvetica", "", 11)
        for label, value in [
            ("Invoice Number", inv["number"]),
            ("Customer",       inv["customer"]),
            ("Customer Email", inv["email"]),
            ("Amount Due",     f"${inv['amount']:,.2f}"),
            ("Invoice Date",   inv["date"]),
            ("Due Date",       inv["due"]),
        ]:
            pdf.cell(60, 8, f"{label}:", ln=False)
            pdf.cell(0,  8, value, ln=True)
        pdf.ln(10)
        pdf.set_font("Helvetica", "I", 10)
        pdf.cell(0, 8, "Thank you for your business.", ln=True, align="C")
        pdf.output(str(pdf_path))
    print(f"Sample invoices ready in {INPUT_INVOICES_DIR}")


create_sample_invoices()

Sample invoices ready in Input\Invoices


In [5]:
def create_erp_excel():
    """Seed ERPDirectory/invoices.xlsx with invoice data if it does not exist."""
    if ERP_EXCEL_PATH.exists():
        print(f"ERP file already exists: {ERP_EXCEL_PATH}")
        return
    df = pd.DataFrame([{
        "Invoice Number": inv["number"],
        "Customer Name":  inv["customer"],
        "Customer Email": inv["email"],
        "Invoice Amount": inv["amount"],
        "Invoice Date":   inv["date"],
        "Due Date":       inv["due"],
        "Status":         "Overdue",
        "Promise Date":   "",
        "Notes":          "",
    } for inv in INVOICE_DATA])
    df.to_excel(ERP_EXCEL_PATH, index=False)
    print(f"ERP Excel created: {ERP_EXCEL_PATH}")


create_erp_excel()

ERP file already exists: ERPDirectory\invoices.xlsx


In [6]:
# ═══════════════════════════════════════════════════════════════
# TOOLS  —  written by the Tools Agent
# ═══════════════════════════════════════════════════════════════

@tool
def send_invoice_to_customer(invoice_number: str) -> str:
    """
    Sends a copy of an invoice PDF to the customer by copying it from the input invoices
    directory to the output directory. Use this tool when a customer has requested a copy
    of their invoice. The invoice_number should match the PDF filename (without extension).
    """
    source_path      = INPUT_INVOICES_DIR / f"{invoice_number}.pdf"
    destination_path = OUTPUT_DIR / f"{invoice_number}.pdf"

    if not source_path.exists():
        return (
            f"Error: Invoice PDF for '{invoice_number}' not found at '{source_path}'. "
            f"Please verify the invoice number is correct."
        )
    try:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(source_path), str(destination_path))
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return (
            f"[{timestamp}] Successfully sent invoice '{invoice_number}'. "
            f"Copy saved to '{destination_path}'."
        )
    except Exception as e:
        return f"Error sending invoice '{invoice_number}': {str(e)}"


@tool
def update_inquiry_status_database(
    invoice_number: str,
    customer_name: str,
    customer_email: str,
    inquiry_type: str,
    promise_date: str = "",
    notes: str = "",
) -> str:
    """
    Records a processed customer inquiry into the SQLite database. Use this tool after
    handling any customer email to log the outcome. The inquiry_type must be either
    'invoice_request' (customer asked for a copy of their invoice) or 'promise_date'
    (customer provided a date by which they promise to pay). Optionally supply a
    promise_date (YYYY-MM-DD) and any notes about the interaction.
    """
    if inquiry_type not in ("invoice_request", "promise_date"):
        return (
            f"Error: inquiry_type must be 'invoice_request' or 'promise_date', "
            f"got '{inquiry_type}'."
        )
    try:
        conn   = sqlite3.connect(str(DB_PATH))
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS invoice_inquiry (
                id             INTEGER PRIMARY KEY AUTOINCREMENT,
                invoice_number TEXT NOT NULL,
                customer_name  TEXT NOT NULL,
                customer_email TEXT NOT NULL,
                inquiry_type   TEXT NOT NULL,
                promise_date   TEXT,
                status         TEXT DEFAULT 'processed',
                email_file     TEXT,
                notes          TEXT,
                processed_at   TEXT NOT NULL
            )
        """)
        processed_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute(
            """
            INSERT INTO invoice_inquiry
                (invoice_number, customer_name, customer_email, inquiry_type,
                 promise_date, status, email_file, notes, processed_at)
            VALUES (?, ?, ?, ?, ?, 'processed', NULL, ?, ?)
            """,
            (
                invoice_number,
                customer_name,
                customer_email,
                inquiry_type,
                promise_date if promise_date else None,
                notes        if notes        else None,
                processed_at,
            ),
        )
        conn.commit()
        conn.close()
        return (
            f"[{processed_at}] Successfully recorded inquiry in database. "
            f"Invoice: '{invoice_number}', Customer: '{customer_name}', "
            f"Type: '{inquiry_type}', Status: 'processed'."
        )
    except Exception as e:
        return f"Error updating inquiry database for invoice '{invoice_number}': {str(e)}"


@tool
def update_erp(
    invoice_number: str,
    promise_date: str,
    customer_name: str = "",
    notes: str = "",
) -> str:
    """
    Updates the ERP Excel file with a customer's payment promise date. Use this tool
    when a customer has provided a date by which they promise to make payment. The
    tool finds the matching invoice row by invoice_number, sets the Promise Date column
    to promise_date (YYYY-MM-DD), and changes the Status to 'Promise Received'.
    Optionally updates the Notes column if notes are provided.
    """
    if not ERP_EXCEL_PATH.exists():
        return f"Error: ERP Excel file not found at '{ERP_EXCEL_PATH}'."
    try:
        df   = pd.read_excel(str(ERP_EXCEL_PATH))
        mask = df["Invoice Number"].astype(str).str.strip() == str(invoice_number).strip()

        if not mask.any():
            return f"Error: Invoice '{invoice_number}' not found in ERP."

        df.loc[mask, "Promise Date"] = promise_date
        df.loc[mask, "Status"]       = "Promise Received"
        if notes:
            df.loc[mask, "Notes"] = notes
        df.to_excel(str(ERP_EXCEL_PATH), index=False)

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return (
            f"[{timestamp}] ERP updated for invoice '{invoice_number}'. "
            f"Promise Date: '{promise_date}', Status: 'Promise Received'."
        )
    except Exception as e:
        return f"Error updating ERP for invoice '{invoice_number}': {str(e)}"

In [7]:
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═
# AGENTS  —  Agent Creation Agent  |  LangChain v1.x / LangGraph
# ═

PARSE_SYSTEM_PROMPT = (
    "You are a customer email classifier for an Accounts Receivable dunning system.\n\n"
    "Analyze the customer email and:\n"
    "1. Identify the invoice number (format INV-XXX), customer name, and customer email.\n"
    "2. Classify as:\n"
    "   - 'invoice_request': customer is asking for a copy of their invoice\n"
    "   - 'promise_date': customer is providing a payment commitment date\n"
    "3. Act based on classification:\n"
    "   - invoice_request: call send_invoice_to_customer, then call update_inquiry_status_database"
    "     with inquiry_type='invoice_request'\n"
    "   - promise_date: extract date (YYYY-MM-DD), call update_inquiry_status_database"
    "     with inquiry_type='promise_date' and the promise_date\n"
    "4. Your FINAL response must be ONLY this JSON (no other text):\n"
    '{"email_type":"...","invoice_number":"...","customer_name":"...","customer_email":"...","promise_date":""}'
)


def create_parse_customer_response_agent():
    """ParseCustomerResponse: classifies emails, calls send-invoice or DB-update tools."""
    llm   = ChatGoogleGenerativeAI(model=MODELS["parse_agent"], google_api_key=GOOGLE_API_KEY, temperature=0)
    tools = [send_invoice_to_customer, update_inquiry_status_database]
    return create_react_agent(llm, tools, prompt=PARSE_SYSTEM_PROMPT)


ERP_SYSTEM_PROMPT = (
    "You are an ERP update agent for an Accounts Receivable system.\n"
    "Update the ERP system with the provided invoice number and promise date "
    "using the update_erp tool. Confirm what was updated."
)


def create_access_update_erp_agent():
    """AccessUpdateERP: updates ERPDirectory/invoices.xlsx with customer promise dates."""
    llm   = ChatGoogleGenerativeAI(model=MODELS["erp_agent"], google_api_key=GOOGLE_API_KEY, temperature=0)
    tools = [update_erp]
    return create_react_agent(llm, tools, prompt=ERP_SYSTEM_PROMPT)

In [8]:
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═
# ORCHESTRATION  —  Orchestrator Agent  |  LangGraph invoke format
# ═

def orchestrate() -> None:
    """Main loop: for each email call ParseCustomerResponse, then AccessUpdateERP if needed."""
    print("=" * 60)
    print("AR DUNNING MULTI-AGENT ORCHESTRATION")
    print("=" * 60)

    parse_agent = create_parse_customer_response_agent()
    erp_agent   = create_access_update_erp_agent()

    email_files = sorted(INPUT_EMAILS_DIR.glob("*.txt"))
    print(f"\nFound {len(email_files)} email(s) to process.\n")

    for email_file in email_files:
        print(f"\n{'─' * 50}")
        print(f"Processing: {email_file.name}")
        print(f"{'─' * 50}")

        try:
            email_content = email_file.read_text(encoding="utf-8")

            # LangGraph agents use messages format
            parse_result = parse_agent.invoke({"messages": [("human", f"Process this customer email:\n\n{email_content}")]})
            output_str   = parse_result["messages"][-1].content
            print(f"\nParse Agent Output:\n{output_str}\n")

            # Extract the JSON block from the agent's final response
            parsed_data = {}
            start = output_str.rfind("{")
            end   = output_str.rfind("}") + 1
            if start >= 0 and end > start:
                try:
                    parsed_data = json.loads(output_str[start:end])
                except json.JSONDecodeError:
                    print("  Warning: could not parse JSON from parse agent output.")

            # If customer gave a promise date, also update the ERP
            if parsed_data.get("email_type") == "promise_date":
                inv_num      = parsed_data.get("invoice_number", "")
                promise_date = parsed_data.get("promise_date", "")
                cust_name    = parsed_data.get("customer_name", "")
                print(f"  → Promise date detected for {inv_num} — calling AccessUpdateERP")
                erp_agent.invoke({
                    "messages": [("human", f"Update ERP for invoice {inv_num} with promise date {promise_date}. Customer: {cust_name}")]
                })

        except Exception as e:
            print(f"  ERROR processing {email_file.name}: {e}")

        # Always move the email to Processed_Emails regardless of outcome
        dest = PROCESSED_EMAILS_DIR / email_file.name
        shutil.move(str(email_file), str(dest))
        print(f"  → Moved {email_file.name} to Processed_Emails/")

    print(f"\n{'=' * 60}")
    print("All emails processed. Orchestration complete.")
    print(f"{'=' * 60}")

In [9]:
def generate_status_report() -> str:
    """Query invoice_inquiry DB and write an Excel status report to Output/."""
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(
        "SELECT invoice_number, customer_name, customer_email, inquiry_type, "
        "promise_date, status, notes, processed_at "
        "FROM invoice_inquiry ORDER BY processed_at",
        conn,
    )
    conn.close()

    df.columns = [
        "Invoice Number", "Customer Name", "Customer Email",
        "Inquiry Type", "Promise Date", "Status", "Notes", "Processed At",
    ]

    report_path = OUTPUT_DIR / f"StatusReport_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Invoice Inquiry", index=False)

        # Auto-fit column widths
        ws = writer.sheets["Invoice Inquiry"]
        for col in ws.columns:
            max_len = max((len(str(cell.value or "")) for cell in col), default=10)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    print(f"Status report saved: {report_path}")
    print(df.to_string(index=False))
    return str(report_path)

In [10]:
def reset_database() -> None:
    """Reset SQLite DB, ERP Excel, and restore processed emails — for re-running tests."""
    print("Resetting system for testing...")

    conn   = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM invoice_inquiry")
    try:
        cursor.execute("DELETE FROM sqlite_sequence WHERE name='invoice_inquiry'")
    except sqlite3.OperationalError:
        pass  # sqlite_sequence only exists after the first insert
    conn.commit()
    conn.close()

    try:
        df = pd.read_excel(ERP_EXCEL_PATH)
        df["Promise Date"] = ""
        df["Status"]       = "Overdue"
        df["Notes"]        = ""
        df.to_excel(ERP_EXCEL_PATH, index=False)
        print("ERP Excel reset.")
    except Exception as e:
        print(f"Warning: could not reset ERP Excel: {e}")

    moved = 0
    for f in PROCESSED_EMAILS_DIR.glob("*.txt"):
        shutil.move(str(f), str(INPUT_EMAILS_DIR / f.name))
        moved += 1
    if moved:
        print(f"Moved {moved} email(s) back to Input_Emails/")

    print("Reset complete — system ready for testing.")

## Execution

Run the cell below to process all emails end-to-end.
Call `reset_database()` first to restore a clean state between test runs.

In [11]:
# ── Optional reset before a fresh run ──────────────────────────
reset_database()

# ── Run the AR Dunning pipeline ─────────────────────────────────
orchestrate()
generate_status_report()

Resetting system for testing...
ERP Excel reset.
Reset complete — system ready for testing.
AR DUNNING MULTI-AGENT ORCHESTRATION


C:\Users\USER\AppData\Local\Temp\ipykernel_21092\1103674057.py:86: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(llm, tools, prompt=PARSE_SYSTEM_PROMPT)
C:\Users\USER\AppData\Local\Temp\ipykernel_21092\1103674057.py:100: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(llm, tools, prompt=ERP_SYSTEM_PROMPT)



Found 5 email(s) to process.


──────────────────────────────────────────────────
Processing: email_001.txt
──────────────────────────────────────────────────

Parse Agent Output:
{"email_type": "invoice_request", "invoice_number": "INV-001", "customer_name": "ACME Corp", "customer_email": "john.smith@acmecorp.com", "promise_date": ""}

  → Moved email_001.txt to Processed_Emails/

──────────────────────────────────────────────────
Processing: email_002.txt
──────────────────────────────────────────────────

Parse Agent Output:
{"email_type": "promise_date", "invoice_number": "INV-002", "customer_name": "Sarah Jones", "customer_email": "sarah.jones@techstartup.io", "promise_date": "2026-06-20"}

  → Promise date detected for INV-002 — calling AccessUpdateERP
  ERROR processing email_002.txt: Error calling model 'gemini-3.1-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.1-flash is not found for API version v1beta, or is not supported for generate

'Output\\StatusReport_20260530_162202.xlsx'